# 02 — Modeling: Seoul Bike Demand

## ขั้น 1 — นิยามปัญหา

**หน่วยของข้อมูล:** 1 แถว = 1 ชั่วโมง ของ **ทั้งระบบ** Seoul Bike (ยอดรวมทุกสถานีในเมือง) ตั้งแต่ 1 ธ.ค. 2017 – 30 พ.ย. 2018

| หัวข้อ | **classification (ใช้ตอนนี้ — default)** | regression (ถ้าอาจารย์อนุญาต) |
|---|---|---|
| ทำนายอะไร | ชั่วโมงนี้เป็นชั่วโมง **"ความต้องการสูง"** หรือไม่ (`high_demand` = 1/0) | จำนวนจักรยานที่ถูกเช่า (คัน/ชม.) |
| นิยาม "สูง" | ยอดเช่า ≥ quantile ที่ 0.75 ของ **train** (top 25%) — ค่า threshold คำนวณในขั้น 5 | — |
| ใช้ทำอะไร | แจ้งเตือนล่วงหน้าให้ผู้ดูแลระบบเตรียมจักรยาน/จัดคนงานกระจายจักรยานในชั่วโมงที่คาดว่าจะมีคนเช่ามาก | วางแผนจำนวนจักรยานที่ต้องพร้อมใช้ล่วงหน้า |
| input | สภาพอากาศ (อุณหภูมิ, ความชื้น, ลม, ทัศนวิสัย, dew point, แสงแดด, ฝน, หิมะ) + เวลา (ชั่วโมง, วันในสัปดาห์, เดือน) + ฤดู + วันหยุด | เหมือนกัน |
| เมตริกหลัก | **F1 ของคลาส "สูง"** (`pos_label=1`) + **PR-AUC** (average precision) | **MAE** (คัน/ชม.) |
| เมตริกรอง | precision, recall, ROC-AUC, confusion matrix | RMSE, R² |

**ทำไมไม่ใช้ accuracy เป็นเมตริกหลัก:** คลาส "สูง" มีแค่ ~25% ถ้าโมเดลทายว่า "ไม่สูง" ทุกชั่วโมงก็ได้ accuracy ≈ 75% ทั้งที่ไม่มีประโยชน์เลย
F1 ของคลาส 1 บังคับให้โมเดลต้องจับชั่วโมงที่สูงได้จริง (recall) และไม่แจ้งเตือนมั่ว (precision) ส่วน PR-AUC วัดคุณภาพการจัดอันดับความน่าจะเป็นโดยไม่ผูกกับ threshold 0.5

**ขอบเขตที่ประกาศ**
- โมเดลใช้กับ **ชั่วโมงที่ระบบเปิดให้บริการ** (`Functioning Day = Yes`) เท่านั้น — ชั่วโมงที่ระบบปิดยอดเช่าเป็น 0 เสมอและรู้ล่วงหน้าอยู่แล้ว (รายละเอียดใน `01_eda.ipynb` ขั้น 4)
- input ทุกตัวเป็นข้อมูลที่ **รู้ล่วงหน้าได้** (พยากรณ์อากาศ + ปฏิทิน) → **ไม่ใช้** ยอดเช่าของชั่วโมงก่อนหน้า (lag) เพราะนั่นคือการใช้ target ทำนาย target
- โจทย์คือ "อากาศและเวลาแบบนี้ → ความต้องการสูงไหม" ไม่ใช่ time-series forecasting → จึง split ตามวันแบบสุ่ม (ขั้น 3) ไม่ใช่ตัดตามเวลา
- ทำนายระดับทั้งเมือง ไม่ใช่รายสถานี (ข้อมูลไม่มีรายสถานี)

> ⚠️ กฎหลักของ notebook นี้
> - preprocessing ทุกขั้นที่เรียนจากข้อมูล (imputer, scaler, encoder, selector, PCA) อยู่ใน `Pipeline` — CV จะ fit ใหม่ทุก fold
> - เลือกโมเดล/ไฮเปอร์พารามิเตอร์ด้วย `GroupKFold` บน train (group = วันที่) และส่ง `groups=` ทุกครั้ง
> - **ห้ามดู test จนเลือกโมเดลเสร็จ** → ประเมิน test ครั้งเดียวตอนท้าย

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import config
# TODO: from src.data import load_raw, load_split  (เมื่อเขียน src/data.py เสร็จ)
print("TASK =", config.TASK, "| RANDOM_STATE =", config.RANDOM_STATE)

## 1. โหลด split + เตรียม target
- TODO: `load_split()` + groups (วันที่) ของ train
- **classification**: `high_demand = y >= threshold`
  - ⚠️ threshold คำนวณจาก **y_train เท่านั้น** แล้วใช้ค่าเดียวกันกับ y_test
  - ⚠️ ตัด `Rented Bike Count` ออกจาก feature (ไม่งั้น leak)
  - ⚠️ ดู class balance หลังตั้ง threshold
- **regression**: target = `Rented Bike Count` ตรงๆ
- ⚠️ คอลัมน์วันที่ใช้เป็น groups ไม่ใช่ feature ตรงๆ (ถ้าจะใช้ ให้สกัดเป็น weekday/month เอง)

In [ ]:
# TODO

## 2. Preprocessing pipeline
- TODO: `ColumnTransformer` — ตัวเลข (scale) / หมวดหมู่ (one-hot)
- ⚠️ Hour เป็นตัวเลขหรือหมวดหมู่? (หรือ cyclic sin/cos) — ให้เหตุผล
- ⚠️ นำการตัดสินใจจาก 01 (outlier, Functioning Day) มาใช้ให้สอดคล้อง

In [ ]:
# TODO

## 3. Feature selection

| วิธี | classification | regression |
|---|---|---|
| Filter | Chi² (⚠️ ต้องเป็นค่าไม่ติดลบ) / mutual_info_classif | correlation / `f_regression` / mutual_info_regression (⚠️ Chi² ใช้ไม่ได้) |
| Wrapper | RFE | RFE |
| Embedded | L1 Logistic Regression | Lasso |

- TODO: ทำทั้ง 3 วิธี **ภายใน Pipeline + CV** แล้วเทียบ feature ที่ได้
- ⚠️ ถ้าเลือก feature บน train ทั้งก้อนก่อน CV → ผล CV มองโลกในแง่ดีเกินจริง

In [ ]:
# TODO: Filter

In [ ]:
# TODO: Wrapper (RFE)

In [ ]:
# TODO: Embedded

## 4. PCA
- TODO: standardize ก่อน → plot explained variance (cumulative)
- TODO: เลือกจำนวน component + เทียบผลโมเดล มี/ไม่มี PCA
- ⚠️ PCA อยู่ใน Pipeline หลัง scaler

In [ ]:
# TODO

## 5. เทียบ ≥ 4 โมเดล (GroupKFold บน train)
- TODO: เลือก ≥ 4 โมเดลที่ต่างแนวคิด + เหตุผล
- เมตริก
  - **classification**: confusion matrix, precision, recall, F1 (⚠️ ระบุ `average`), ROC-AUC
  - **regression**: MAE (หลัก), RMSE, R² · ⚠️ MAPE ใช้ตรงๆ ไม่ได้ เพราะมียอด = 0
- ⚠️ ใช้ `GroupKFold(groups=วันที่)` — `KFold` ธรรมดาจะรั่วชั่วโมงวันเดียวกัน
- TODO: ตาราง mean ± std ของแต่ละโมเดล

In [ ]:
# TODO

## 6. ตาราง resource
- TODO: ต่อโมเดล — ขนาดไฟล์ joblib (KB), เวลา fit (s), latency ต่อ 1 ตัวอย่าง (ms)
- ⚠️ วัด latency หลายรอบแล้วเฉลี่ย

In [ ]:
# TODO

## 7. ประเมิน test (ครั้งเดียว)
- TODO: fit โมเดลที่เลือกบน train ทั้งหมด → ประเมิน test
- ⚠️ ถ้าผล test แย่กว่า CV มาก ให้อภิปราย อย่ากลับไปจูนด้วย test

In [ ]:
# TODO

## 8. ทำนายข้อมูลสมมติ + error analysis
- TODO: สร้างข้อมูลสมมติ 2–3 แถว (สถานการณ์ต่างกัน) แล้วทำนาย
- TODO: ดูตัวอย่างที่ผิดมาก — เกิดช่วงไหน (ชั่วโมง/ฤดู/สภาพอากาศ)?

In [ ]:
# TODO

## 9. สรุป
- TODO